# Laboratorio 1 · Entorno y primeros casos comprobados

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udd-ia-generativa-2026/curso-ia/blob/main/notebooks/lab01.ipynb)

**Formativo, sin nota.** Trabajo individual en Google Colab: instalas el código del curso, ejecutas un caso común en dos modelos y entregas tu evidencia en Canvas.

Al terminar tendrás: una llamada a un modelo escrita por ti, casos ejecutados en dos modelos, una revisión de cada respuesta y una tabla con tokens, latencia y costo.

Antes de empezar, abre **Secretos** (el ícono de llave del panel izquierdo) y agrega tu clave de Gemini como `GOOGLE_API_KEY` o `GEMINI_API_KEY` (el nombre que usa el botón de Colab) y, si el docente te los entregó, `CURSO_API_KEY` y `CURSO_BASE_URL` del servidor del curso, con acceso para este notebook. El servidor del curso es un respaldo que solo funciona durante las clases de los viernes; Gemini es la ruta principal. No escribas claves en las celdas ni imprimas `os.environ`.

Al instalar, pip puede avisar un conflicto de `google-auth` con `google-colab`: es esperado y no afecta al laboratorio. Si el docente publica una versión nueva durante la clase, usa **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar desde la primera celda.

In [ ]:
import os
import sys

# En Colab se instala el código del curso; en un repositorio con entorno propio ya está instalado.
if "COLAB_RELEASE_TAG" in os.environ:
    %pip install -q "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@main"
    # Reinstala solo el paquete del curso, por si quedó una copia anterior con la misma versión.
    %pip install -q --force-reinstall --no-deps "curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@main"
    if "curso_ia" in sys.modules:
        print("El paquete ya estaba cargado: usa Entorno de ejecución → Reiniciar sesión y vuelve a ejecutar desde aquí.")


In [ ]:
import os
from pathlib import Path

import pandas as pd
from pydantic_ai import Agent

from curso_ia import (
    apply_review, blocked, create_model, lab_cases, load_config, parse_cases, review_template, run_case, save_run,
    status,
)

# Modelo de comparación: texto libre con la forma proveedor:modelo. Opciones recomendadas:
#   "google:gemma-4-26b-a4b-it"  Gemma 4, de pesos abiertos, gratis con tu clave de Gemini; puede tardar.
#   "curso:course-chat"          servidor del curso, solo durante las clases de los viernes;
#                                requiere los secretos CURSO_API_KEY y CURSO_BASE_URL.
COMPARISON_MODEL = "google:gemma-4-26b-a4b-it"
os.environ["COMPARISON_MODEL"] = COMPARISON_MODEL

config = load_config()
lab = Path("lab01")  # aquí quedan la evidencia y el archivo para Canvas
if config.mode == "simulated":
    print("MODO SIMULADO: respuestas simuladas; no acredita acceso a ningún modelo.")
for row in status(config):
    state = "listo" if row["ready"] else "falta: " + ", ".join(row["missing"])
    print(f"{row['role']:<11} {row['model']:<32} {state}")


La lista muestra qué modelo atiende cada rol (`proveedor:modelo`) y, si falta algo, el **nombre** de la variable; nunca su valor. `generation` usa Gemini y `comparison` usa el modelo que escribiste en `COMPARISON_MODEL`, en la celda anterior. Por defecto es Gemma 4, un modelo de pesos abiertos de Google que la API de Gemini sirve gratis, así que la comparación es entre un modelo comercial y uno abierto. Durante la clase puedes usar en su lugar el servidor del curso con `curso:course-chat`; fuera de clase ese servidor no responde. Si cambias `COMPARISON_MODEL`, vuelve a ejecutar esa celda. Con una clave propia también puedes cambiar `generation` con la variable `GENERATION_MODEL`.

## 1. Tu primera llamada

Un `Agent` de PydanticAI recibe un modelo y una instrucción; `run` envía la consulta y devuelve la respuesta (`output`) y el consumo (`usage`), con los tokens de entrada y de salida que determinan el costo. La celda usa el primer perfil que esté listo.

`run` es asíncrona: en un notebook se escribe `await agent.run(...)` directo en la celda, porque Jupyter y Colab ya tienen un bucle de eventos (asyncio) en marcha. El editor de Colab puede subrayar `await` con el aviso «allowed only within async function»: es un aviso del editor y la celda se ejecuta igual. `run_sync` no sirve aquí, justamente porque ese bucle ya está corriendo.

In [ ]:
ready = [row["role"] for row in status(config) if row["ready"]]
role = ready[0] if ready else "generation"  # el primer perfil listo; si ninguno, muestra qué falta
print("Perfil:", role, config.models[role])
model = create_model(config.models[role], config)
agent = Agent(model, instructions="Responde en una sola frase.")
result = await agent.run("¿Qué es un token en un modelo de lenguaje?")
print(result.output)
print(result.usage)


## 2. Los casos

El caso común del curso viene en el paquete: la regla de tarifas por duelo del caso Air Canada en cuatro consultas, cada una con su respuesta esperada. Ejecútalo sin cambios.

Mínimo para hoy: el caso común en ambos perfiles; si el servidor del curso no está disponible, basta con Gemini y el bloqueo registrado. Extensión: en la celda siguiente, reemplaza el ejemplo de `MY_TASK` por dos o tres casos de una tarea que te interese (una instrucción y, por caso, una entrada y la respuesta que esperas), cambia `USE_MY_TASK` a `True` y vuelve a ejecutar desde aquí. Usa datos públicos o sintéticos.

In [ ]:
MY_TASK = {
    "instruction": 'Clasifica el mensaje como "reclamo", "consulta" o "felicitación". Responde solo con la categoría.',
    "cases": [
        {"id": "ejemplo-1", "input": "¿A qué hora abre la sucursal de Providencia el sábado?", "expected": "consulta"},
        {"id": "ejemplo-2", "input": "Me cobraron dos veces la misma compra y nadie responde.", "expected": "reclamo"},
    ],
}
USE_MY_TASK = False  # extensión: True para ejecutar tus casos

cases = parse_cases(MY_TASK, "my-task") if USE_MY_TASK else lab_cases(1)
if cases.is_example:
    print("Aún usas los casos de ejemplo: reemplázalos por los de tu tarea.")
print("Instrucción:", cases.instruction)
pd.DataFrame([vars(case) for case in cases.cases])


## 3. Ejecutar en ambos perfiles

Cada caso se envía por separado, sin mensajes anteriores. Si un perfil no está configurado o falla, se registra el bloqueo y se sigue con el otro.

Si un caso queda en `error`, las columnas `error_type` y `http_status` dicen por qué; un 429 es el límite de solicitudes por minuto del nivel gratuito de Gemini: espera un minuto y vuelve a ejecutar. Un 503 indica alta demanda en Google: espera unos minutos y vuelve a ejecutar. Evita cambiar a un modelo Flash sin «Lite»: en el nivel gratuito solo admiten 20 solicitudes al día.

`SETTINGS` ajusta cómo responde el modelo, por ejemplo la temperatura o el máximo de tokens; ver [ajustes de modelo en PydanticAI](https://ai.pydantic.dev/agents/#model-run-settings) y la [lista de campos de `ModelSettings`](https://ai.pydantic.dev/api/settings/) (inglés).

Extensión: sube `REPETITIONS` a 2 o 3 para ver cuánto varían la respuesta, los tokens y la latencia.

In [ ]:
from tqdm.auto import tqdm

REPETITIONS = 1
SETTINGS = {}  # por ejemplo {"temperature": 0.2}; vacío usa el valor de cada proveedor

rows = []
progress = tqdm(total=len(config.models) * len(cases.cases) * REPETITIONS, desc="Consultas")
for profile, ref in config.models.items():
    try:
        agent = Agent(create_model(ref, config), instructions=cases.instruction, model_settings=SETTINGS)
    except Exception as error:
        rows += [blocked(profile, ref, case, config, 1, error) for case in cases.cases]
        progress.update(len(cases.cases) * REPETITIONS)
        continue
    for case in cases.cases:
        for repetition in range(1, REPETITIONS + 1):
            progress.set_postfix_str(f"{profile} · {case.id}")
            rows.append(await run_case(agent, case, profile=profile, ref=ref, config=config, repetition=repetition))
            progress.update()
progress.close()

columns = [
    "profile", "case", "repetition", "status", "input_tokens", "output_tokens", "latency_ms",
    "estimated_cost_usd", "error_type", "http_status",
]
pd.DataFrame(rows).reindex(columns=columns)


### Leer las respuestas

La tabla anterior trae las mediciones. Aquí se ven las respuestas completas, agrupadas por caso: primero la respuesta esperada y luego lo que respondió cada perfil. Léelas antes de revisarlas en la sección 5.

In [ ]:
from IPython.display import Markdown, display

expected = {case.id: case.expected for case in cases.cases}
display(Markdown(f"**Instrucción:** {cases.instruction}"))

for row in rows:
    if row["status"] != "ok" and row["case"] == cases.cases[0].id:
        reason = ", ".join(row.get("missing", [])) or row.get("error_type", row["status"])
        display(Markdown(f"> **{row['profile']}** ({row['model']}) no respondió: {reason}."))

for case in cases.cases:
    display(Markdown(f"---\n### {case.id}\n**Esperada:** {expected[case.id]}"))
    for row in rows:
        if row["case"] != case.id or row["status"] != "ok":
            continue
        cost = f" · {row['estimated_cost_usd']} USD" if row.get("estimated_cost_usd") is not None else ""
        display(Markdown(
            f"#### {row['profile']} · {row['model']} (repetición {row['repetition']})\n\n{row['response']}\n\n"
            f"<sub>{row['input_tokens']} tokens de entrada · {row['output_tokens']} de salida · "
            f"{row['latency_ms']} ms{cost}</sub>"
        ))


## 4. Guardar la ejecución

La ejecución completa queda en un JSON dentro de `lab01/evidence/`, que no depende de las salidas del notebook: `cases.json` para el caso común y `my-task.json` para tus casos. Si vuelves a ejecutar, se reemplaza la ejecución anterior del mismo conjunto, con su revisión.

In [ ]:
evidence_path = save_run(rows, cases, config, lab / "evidence", lab=1, settings=SETTINGS)
print("Ejecución guardada en", evidence_path)


## 5. Revisar cada respuesta

Ejecuta la celda siguiente: imprime un diccionario `REVIEW` con una entrada por respuesta y, encima de cada una, la respuesta esperada. Cópialo en la celda de abajo y completa cada par: `True` si la respuesta cumple o `False` si no, y un motivo en una frase (qué coincide, qué falta o qué se agregó sin respaldo). Relee las respuestas en «Leer las respuestas». Una respuesta incorrecta es tan útil como una correcta si explicas por qué.

In [ ]:
print(review_template(evidence_path))


In [ ]:
# Pega aquí el diccionario REVIEW y complétalo; puedes corregirlo y volver a ejecutar.
REVIEW = {}

evidence = apply_review(evidence_path, REVIEW, config)
print("Revisión completa:", evidence["review_complete"])


Con la revisión completa, la celda siguiente resume cada perfil.

In [ ]:
numeric = ["input_tokens", "output_tokens", "latency_ms", "estimated_cost_usd"]
reviewed = pd.DataFrame(evidence["results"]).reindex(columns=["profile", "status", "passed", *numeric])
reviewed[numeric] = reviewed[numeric].apply(pd.to_numeric, errors="coerce")
reviewed.groupby("profile").agg(
    responses=("status", lambda s: (s == "ok").sum()),
    passed=("passed", lambda s: (s == True).sum()),  # noqa: E712
    input_tokens=("input_tokens", "mean"),
    output_tokens=("output_tokens", "mean"),
    latency_ms=("latency_ms", "median"),
    cost_usd=("estimated_cost_usd", lambda s: s.sum(min_count=1)),
)


Lee la tabla con cuidado: con pocos casos no se puede concluir qué modelo es mejor; es una primera medición. El costo es una estimación: la fuente de cada precio queda en el JSON, y en el nivel gratuito de Gemini no pagas el precio de lista que se muestra.

## 6. Entregar y explicar

La celda siguiente comprime `lab01` con la ejecución y la revisión, y lo descarga. Descarga también este notebook con sus salidas (**Archivo → Descargar → Descargar .ipynb**). Los archivos de Colab se borran al cerrar la sesión: descarga antes de salir.

En la tarea del laboratorio 1 en Canvas, sube ambos archivos y explica en pocas líneas un caso que falló o un costo que te sorprendió, y qué cambiarías. Si usaste un asistente, declara su uso según las reglas del curso. Si ya tienes GitHub, también puedes guardar estos archivos en un repositorio propio; no es obligatorio.

Las parejas se forman después de esta clase; tu tarea y tus casos te servirán para proponer el tema del proyecto.

In [ ]:
import shutil

archive = shutil.make_archive("lab01-submission", "zip", lab.parent, lab.name)
if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import files
    files.download(archive)
print("Archivo para Canvas:", archive)
